# Task 5: Bubble Chart — App Size vs Average Rating (Bubble Size = Installs)

**Requirements:**
- Bubble chart: **x = app size (MB)**, **y = average rating**, **bubble size = installs**
- Filters:
  - Rating > 3.5
  - Category in: Game, Beauty, Business, Comics, Communication, Dating, Entertainment, Social, Events
  - Reviews > 500
  - App name must **not contain** the letter `s`
  - Sentiment subjectivity (from user reviews) > 0.5
  - Installs > 50,000
- Highlight the **Game** category in **pink**
- Category label translation: **Beauty → Hindi**, **Business → Tamil**, **Dating → German** (all three pass the category filter this time, so all three translations actually render)
- Display rule: only visible between **5 PM – 7 PM IST**

### Data note: sentiment subjectivity
`googleplaystore.csv` has no sentiment data — that lives in `user_reviews.csv`, one row per review. We compute each app's **average `Sentiment_Subjectivity` across all its reviews**, then join that back onto the app table before filtering.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

## 1. Load and clean both datasets

In [2]:
df = pd.read_csv('googleplaystore.csv')
df = df.drop_duplicates(subset='App', keep='first')
df = df[~df['Category'].astype(str).str.contains(r'^\d', regex=True, na=False)]

def parse_size(size):
    if pd.isna(size):
        return np.nan
    size = str(size).strip()
    if size == 'Varies with device' or size == '':
        return np.nan
    if size.endswith('M'):
        return float(size[:-1])
    if size.endswith('k') or size.endswith('K'):
        return float(size[:-1]) / 1024.0
    try:
        return float(size)
    except ValueError:
        return np.nan
df['Size_MB'] = df['Size'].apply(parse_size)

df['Installs_Num'] = pd.to_numeric(
    df['Installs'].astype(str).str.replace(',', '', regex=False).str.replace('+', '', regex=False),
    errors='coerce'
)
df['Reviews_Num'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

# Reviews dataset -> average sentiment subjectivity per app
reviews = pd.read_csv('user_reviews.csv')
subjectivity = reviews.groupby('App')['Sentiment_Subjectivity'].mean().reset_index()
subjectivity.columns = ['App', 'Avg_Subjectivity']

merged = df.merge(subjectivity, on='App', how='left')
merged[['App','Category','Size_MB','Rating','Installs_Num','Reviews_Num','Avg_Subjectivity']].head()

,App,Category,Size_MB,Rating,Installs_Num,Reviews_Num,Avg_Subjectivity
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,19.0,4.1,10000,159,NaN
1,Coloring book moana,ART_AND_DESIGN,14.0,3.9,500000,967,0.64154
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,8.7,4.7,5000000,87510,NaN
3,Sketch - Draw & Paint,ART_AND_DESIGN,25.0,4.5,50000000,215644,NaN
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,2.8,4.3,100000,967,NaN


## 2. Apply filters

In [3]:
target_categories = ['GAME','BEAUTY','BUSINESS','COMICS','COMMUNICATION','DATING','ENTERTAINMENT','SOCIAL','EVENTS']

name_mask = ~merged['App'].astype(str).str.contains('s', case=False, na=False)

mask = (
    (merged['Rating'] > 3.5) &
    merged['Category'].isin(target_categories) &
    (merged['Reviews_Num'] > 500) &
    name_mask &
    (merged['Avg_Subjectivity'] > 0.5) &
    (merged['Installs_Num'] > 50_000) &
    merged['Size_MB'].notna()
)

filtered = merged[mask].copy()
print(f"Apps after filtering: {len(filtered)}")
filtered['Category'].value_counts()

Apps after filtering: 23


Category
GAME             11
DATING            5
BUSINESS          2
COMMUNICATION     2
ENTERTAINMENT     2
SOCIAL            1
Name: count, dtype: int64

## 3. Category label translation + highlight color mapping

In [4]:
category_translations = {
    'BEAUTY': 'सौंदर्य (Beauty)',       # Hindi for Beauty
    'BUSINESS': 'வணிகம் (Business)',    # Tamil for Business
    'DATING': 'Partnersuche (Dating)'   # German for Dating
}

def display_label(cat):
    return category_translations.get(cat, cat)

filtered['Display_Category'] = filtered['Category'].apply(display_label)

# Color mapping: Game highlighted pink, everything else gets a distinct color from a palette
palette = ['#4C9AFF', '#3DDC97', '#FFD166', '#B084F5', '#06D6A0', '#EF476F', '#FF8C42', '#8CA0B3']
other_cats = [c for c in filtered['Category'].unique() if c != 'GAME']
color_map = {'GAME': '#FF3FA4'}  # pink highlight
for i, c in enumerate(other_cats):
    color_map[c] = palette[i % len(palette)]

color_map

{'GAME': '#FF3FA4',
 'BUSINESS': '#4C9AFF',
 'COMMUNICATION': '#3DDC97',
 'DATING': '#FFD166',
 'ENTERTAINMENT': '#B084F5',
 'SOCIAL': '#06D6A0'}

## 4. Bubble chart (Plotly)

In [5]:
fig = go.Figure()

for cat in filtered['Category'].unique():
    sub = filtered[filtered['Category'] == cat]
    fig.add_trace(go.Scatter(
        x=sub['Size_MB'],
        y=sub['Rating'],
        mode='markers',
        name=display_label(cat),
        marker=dict(
            size=sub['Installs_Num'],
            sizemode='area',
            sizeref=2.*filtered['Installs_Num'].max()/(60.**2),
            sizemin=4,
            color=color_map[cat],
            line=dict(width=1, color='white'),
            opacity=0.8
        ),
        text=sub['App'],
        hovertemplate='<b>%{text}</b><br>Size: %{x} MB<br>Rating: %{y}<br>Installs: %{marker.size:,}<extra></extra>'
    ))

fig.update_layout(
    title='App Size vs Average Rating (Bubble Size = Installs)<br><sup>Game category highlighted in pink</sup>',
    xaxis_title='App Size (MB)',
    yaxis_title='Average Rating',
    legend=dict(orientation='h', y=1.15, x=0.5, xanchor='center'),
    height=620,
    template='plotly_white',
    margin=dict(t=120, b=60)
)

fig.show()

## 5. Export for the combined dashboard

In [6]:
export_cols = ['App','Category','Display_Category','Size_MB','Rating','Installs_Num','Avg_Subjectivity']
filtered[export_cols].to_csv('task5_bubble_data.csv', index=False)
print('Saved: task5_bubble_data.csv')

Saved: task5_bubble_data.csv


### Note on the 5PM–7PM IST display rule
Same pattern as Tasks 1–4: dashboard-level display rule, implemented with the same IST time-check logic in the combined `dashboard.html`.